# kpi

In [0]:
%sql
-- 1. Monthly Revenue and Monthly Revenue Growth (%)
WITH monthly_revenue AS (
  SELECT
    year(posting_date) AS year,
    month(posting_date) AS month,
    SUM(credit_amount) AS total_revenue
  FROM pl_workforce_catalog.gold.master_cube
  WHERE posting_date IS NOT NULL
  GROUP BY year(posting_date), month(posting_date)
)
SELECT
  year,
  month,
  total_revenue,
  (total_revenue - lag(total_revenue) OVER (ORDER BY year, month)) / NULLIF(lag(total_revenue) OVER (ORDER BY year, month), 0) * 100 AS mon_growth_pct
FROM monthly_revenue
ORDER BY year, month

In [0]:
%sql
-- 2. Cost of Sales by Month and Company (Direct Costs only)
SELECT
  year(posting_date) AS year,
  month(posting_date) AS month,
  company_id,
  company_name,
  SUM(CASE WHEN category = 'Direct Costs' THEN debit_amount ELSE 0 END) AS cost_of_sales
FROM pl_workforce_catalog.gold.master_cube
WHERE posting_date IS NOT NULL
  AND category = 'Direct Costs'
GROUP BY year, month, company_id, company_name
ORDER BY year, month, company_id

In [0]:
%sql
-- 3. Monthly Gross Profit Margin (%)
SELECT
  year(posting_date) AS year,
  month(posting_date) AS month,
  company_id,
  SUM(credit_amount) AS total_revenue,
  SUM(CASE WHEN category = 'Direct Costs' THEN debit_amount ELSE 0 END) AS cost_of_sales,
  (SUM(credit_amount) - SUM(CASE WHEN category = 'Direct Costs' THEN debit_amount ELSE 0 END)) / NULLIF(SUM(credit_amount), 0) * 100 AS gross_profit_margin_pct
FROM pl_workforce_catalog.gold.master_cube
WHERE posting_date IS NOT NULL
GROUP BY year, month, company_id
ORDER BY year, month, company_id

In [0]:
%sql
-- 4. Operating Expense Breakdown by Account Range and Company
SELECT
  company_id,
  company_name,
  account_id,
  account_type,
  category,
  SUM(debit_amount) AS total_expense
FROM pl_workforce_catalog.gold.master_cube
WHERE debit_amount > 0
GROUP BY company_id, company_name, account_id, account_type, category
ORDER BY company_id, account_id;

In [0]:
%sql
-- 5. Average Compensation by Position
SELECT
  company_id,
  position,
  AVG(base_salary + bonus + overtime_pay + commission) AS avg_total_compensation
FROM pl_workforce_catalog.gold.master_cube
GROUP BY company_id, position


In [0]:
%sql
-- 6. Net Profit
SELECT
  year(posting_date) AS year,
  month(posting_date) AS month,
  SUM(credit_amount) - SUM(debit_amount) AS net_profit
FROM pl_workforce_catalog.gold.master_cube
GROUP BY year, month

In [0]:
%sql
-- 7. Overtime & Bonus Analysis by Department
SELECT
  department_id,
  department_name,
  SUM(overtime_pay) AS total_overtime,
  SUM(bonus) AS total_bonus,
  SUM(overtime_pay + bonus) AS total_variable_compensation,
  SUM(base_salary) AS total_base_salary,
  SUM(overtime_pay) / NULLIF(SUM(base_salary), 0) AS overtime_to_base_salary_ratio
FROM pl_workforce_catalog.gold.master_cube
GROUP BY department_id, department_name

In [0]:
%sql
-- 8. Total Department Cost (Payroll Expenses Only)
SELECT
  department_id,
  department_name,
  SUM(base_salary + bonus + overtime_pay + commission) AS total_department_cost
FROM pl_workforce_catalog.gold.master_cube
WHERE posting_date IS NOT NULL
GROUP BY department_id, department_name

In [0]:
%sql
-- 9. Headcount Distribution by Department - Active employee count By Department
SELECT
  department_id,
  department_name,
  COUNT(DISTINCT employee_id) AS active_employee_count
FROM pl_workforce_catalog.gold.master_cube
WHERE is_active = 'TRUE'
GROUP BY department_id, department_name
ORDER BY department_id

In [0]:
%sql
-- 10. Payroll Cost % of Revenue (Yearly and Monthly)
SELECT
  company_id,
  year(posting_date) AS year,
  month(posting_date) AS month,
  SUM(base_salary + bonus + overtime_pay + commission) AS payroll_cost,
  SUM(credit_amount) AS company_revenue,
  (SUM(base_salary + bonus + overtime_pay + commission) / NULLIF(SUM(credit_amount), 0)) * 100 AS payroll_cost_pct_revenue
FROM pl_workforce_catalog.gold.master_cube
WHERE posting_date IS NOT NULL
GROUP BY company_id, year, month
ORDER BY company_id, year, month